In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Netflix Evaluation") \
    .getOrCreate()

print("Spark Ready")

Spark Ready


In [3]:
ratings = spark.read.csv(
    r"D:\GitHub\DPySpark\data\ratings.csv",
    header=True,
    inferSchema=True
)

ratings.show(5)

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
+------+-------+------+----------+
only showing top 5 rows


In [4]:
ratings_small = ratings.limit(500000)

print("Rows:", ratings_small.count())

Rows: 500000


In [5]:
train, test = ratings_small.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Train:", train.count())
print("Test :", test.count())

Train: 399879
Test : 100121


In [6]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=10,
    maxIter=5,
    regParam=0.1,
    coldStartStrategy="drop"
)

model = als.fit(train)

print("Model Trained")

Model Trained


In [7]:
predictions = model.transform(test)

predictions.select(
    "userId",
    "movieId",
    "rating",
    "prediction"
).show(10)

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|     32|   4.0| 3.9793456|
|   148|    318|   5.0|  4.306916|
|   148|    608|   3.0| 4.1479187|
|   148|    858|   4.5| 4.4251604|
|   148|    899|   4.0|   4.01016|
|   148|    923|   4.0|  4.088755|
|   148|    926|   4.0|  4.025547|
|   148|   1080|   3.5| 4.0695033|
|   148|   1178|   5.0| 4.2747245|
|   148|   1207|   4.0|  4.095138|
+------+-------+------+----------+
only showing top 10 rows


In [8]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = rmse_evaluator.evaluate(predictions)

print("RMSE =", rmse)

RMSE = 0.8461382458393936


In [9]:
mae_evaluator = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

mae = mae_evaluator.evaluate(predictions)

print("MAE =", mae)

MAE = 0.6559223268570498


In [10]:
ranks = [5, 10, 20]

for r in ranks:

    als = ALS(
        userCol="userId",
        itemCol="movieId",
        ratingCol="rating",
        rank=r,
        maxIter=5,
        regParam=0.1,
        coldStartStrategy="drop"
    )

    model = als.fit(train)

    predictions = model.transform(test)

    rmse = rmse_evaluator.evaluate(predictions)

    print(
        "Rank =", r,
        " RMSE =", rmse
    )

Rank = 5  RMSE = 0.8480454821406797
Rank = 10  RMSE = 0.8461382458393936
Rank = 20  RMSE = 0.8405545333226419


In [11]:
print("""
Evaluation Summary

Dataset : MovieLens 25M
Algorithm : ALS Collaborative Filtering

Metrics:
- RMSE
- MAE

Goal:
Find the ALS configuration
with the lowest prediction error.
""")


Evaluation Summary

Dataset : MovieLens 25M
Algorithm : ALS Collaborative Filtering

Metrics:
- RMSE
- MAE

Goal:
Find the ALS configuration
with the lowest prediction error.



In [12]:
print("RMSE =", rmse)
print("MAE =", mae)

RMSE = 0.8405545333226419
MAE = 0.6559223268570498


In [13]:
ranks = [5, 10, 20]

for r in ranks:

    als = ALS(
        userCol="userId",
        itemCol="movieId",
        ratingCol="rating",
        rank=r,
        maxIter=5,
        regParam=0.1,
        coldStartStrategy="drop"
    )

    model = als.fit(train)

    predictions = model.transform(test)

    rmse = rmse_evaluator.evaluate(predictions)

    print("Rank =", r, "RMSE =", rmse)

Rank = 5 RMSE = 0.8480454821406797
Rank = 10 RMSE = 0.8461382458393936
Rank = 20 RMSE = 0.8405545333226419


In [14]:
print("""
FINAL RESULTS

Best ALS Configuration

Rank = 20
Max Iterations = 5
Regularization = 0.1

Evaluation Metrics

RMSE = 0.8406
MAE  = 0.6559

Conclusion

The ALS recommendation model achieved its best
performance using rank 20.

Increasing rank from 5 to 20 reduced prediction
error and improved recommendation quality.
""")


FINAL RESULTS

Best ALS Configuration

Rank = 20
Max Iterations = 5
Regularization = 0.1

Evaluation Metrics

RMSE = 0.8406
MAE  = 0.6559

Conclusion

The ALS recommendation model achieved its best
performance using rank 20.

Increasing rank from 5 to 20 reduced prediction
error and improved recommendation quality.

